In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Create database connection
def create_connection():
    try:
        conn = sqlite3.connect('1-13_nba_data.db.db')
        return conn
    except sqlite3.Error as e:
        print(f"Error connecting to database: {e}")
        return None

# Function to setup the database and add new columns
def setup_database(conn):
    cursor = conn.cursor()
    
    # Create table if it doesn't exist (same as before)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS combined_player_game_data (
            GAME_ID INTEGER,
            TEAM_ID INTEGER,
            TEAM_ABBREVIATION TEXT,
            TEAM_CITY TEXT,
            PLAYER_ID INTEGER,
            PLAYER_NAME TEXT,
            NICKNAME TEXT,
            START_POSITION TEXT,
            COMMENT TEXT,
            MIN TEXT,
            FGM INTEGER,
            FGA INTEGER,
            FG_PCT REAL,
            FG3M INTEGER,
            FG3A INTEGER,
            FG3_PCT REAL,
            FTM INTEGER,
            FTA INTEGER,
            FT_PCT REAL,
            OREB INTEGER,
            DREB INTEGER,
            REB INTEGER,
            AST INTEGER,
            STL INTEGER,
            BLK INTEGER,
            "TO" INTEGER,
            PF INTEGER,
            PTS INTEGER,
            PLUS_MINUS INTEGER,
            E_OFF_RATING REAL,
            OFF_RATING REAL,
            E_DEF_RATING REAL,
            DEF_RATING REAL,
            E_NET_RATING REAL,
            NET_RATING REAL,
            AST_PCT REAL,
            AST_TOV REAL,
            AST_RATIO REAL,
            OREB_PCT REAL,
            DREB_PCT REAL,
            REB_PCT REAL,
            TM_TOV_PCT REAL,
            EFG_PCT REAL,
            TS_PCT REAL,
            USG_PCT REAL,
            E_USG_PCT REAL,
            E_PACE REAL,
            PACE REAL,
            PACE_PER40 REAL,
            POSS INTEGER,
            PIE REAL,
            GAME_DATE_EST TEXT,
            GAME_SEQUENCE INTEGER,
            GAME_STATUS_ID INTEGER,
            GAME_STATUS_TEXT TEXT,
            GAMECODE TEXT,
            HOME_TEAM_ID INTEGER,
            VISITOR_TEAM_ID INTEGER,
            SEASON INTEGER,
            LIVE_PERIOD INTEGER,
            LIVE_PC_TIME TEXT,
            NATL_TV_BROADCASTER_ABBREVIATION TEXT,
            LIVE_PERIOD_TIME_BCAST TEXT,
            WH_STATUS INTEGER,
            REGULAR_DATE TEXT,
            HOME_TEAM TEXT,
            AWAY_TEAM TEXT
        )
    """)
    
    # Check if column exists before adding
    cursor.execute("PRAGMA table_info(combined_player_game_data)")
    columns = [col[1] for col in cursor.fetchall()]
    
    if 'opp_avg_pts_against_pos' not in columns:
        try:
            cursor.execute("""
                ALTER TABLE combined_player_game_data 
                ADD COLUMN opp_avg_pts_against_pos REAL
            """)
            print("Successfully added opp_avg_pts_against_pos column")
        except sqlite3.OperationalError as e:
            print(f"Error adding column: {e}")
            conn.close()
            return None
    
    conn.commit()
    return cursor

# Function to calculate defensive stats using pandas
def calculate_defensive_stats(df):
    # Convert GAME_DATE_EST to datetime for proper comparison
    df['GAME_DATE_EST'] = pd.to_datetime(df['GAME_DATE_EST'])
    
    # Create a column for the opposing team ID
    df['opp_team_id'] = np.where(
        df['TEAM_ID'] == df['HOME_TEAM_ID'],
        df['VISITOR_TEAM_ID'],
        df['HOME_TEAM_ID']
    )
    
    # Sort by season and date for chronological processing
    df = df.sort_values(['SEASON', 'GAME_DATE_EST'])
    
    # Function to calculate rolling average for a group
    def calculate_historic_avg(group):
        # For each row, calculate average of previous games
        group['opp_avg_pts_against_pos'] = (
            group.groupby('opp_team_id')['PTS']
            .shift(1)  # Shift to exclude current game
            .expanding()  # Calculate cumulative average up to previous game
            .mean()
        )
        return group
    
    # Calculate for starters (with position)
    starters = df[df['START_POSITION'].notna() & (df['START_POSITION'] != '')].copy()
    if not starters.empty:
        starters = starters.groupby(['SEASON', 'START_POSITION']).apply(calculate_historic_avg)
    
    # Calculate for bench players (no position)
    bench = df[df['START_POSITION'].isna() | (df['START_POSITION'] == '')].copy()
    if not bench.empty:
        bench = bench.groupby('SEASON').apply(calculate_historic_avg)
    
    # Combine the results
    starters = starters.reset_index(drop=True)
    bench = bench.reset_index(drop=True)
    result_df = pd.concat([starters, bench])
    
    return result_df

# Main function to process data and write back to database
def process_data(conn):
    # Load all data into pandas
    query = "SELECT * FROM combined_player_game_data"
    df = pd.read_sql_query(query, conn)
    
    if df.empty:
        print("No data found in the database")
        return
    
    # Calculate defensive stats
    result_df = calculate_defensive_stats(df)
    
    # Select only the columns we need to update
    update_df = result_df[['GAME_ID', 'PLAYER_ID', 'opp_avg_pts_against_pos']]
    
    # Write back to database
    cursor = conn.cursor()
    
    # Create a temporary table for updates
    update_df.to_sql('temp_update', conn, if_exists='replace', index=False)
    
    # Update the main table from the temporary table
    update_query = """
    UPDATE combined_player_game_data
    SET opp_avg_pts_against_pos = (
        SELECT temp_update.opp_avg_pts_against_pos
        FROM temp_update
        WHERE temp_update.GAME_ID = combined_player_game_data.GAME_ID
        AND temp_update.PLAYER_ID = combined_player_game_data.PLAYER_ID
    )
    """
    cursor.execute(update_query)
    conn.commit()
    
    # Clean up temporary table
    cursor.execute("DROP TABLE IF EXISTS temp_update")
    conn.commit()

# Function to retrieve sample results
def get_sample_results(conn):
    query = """
    SELECT 
        GAME_ID,
        SEASON,
        GAME_DATE_EST,
        TEAM_ABBREVIATION,
        PLAYER_NAME,
        START_POSITION,
        PTS,
        opp_avg_pts_against_pos
    FROM combined_player_game_data
    ORDER BY SEASON, GAME_DATE_EST
    LIMIT 10
    """
    return pd.read_sql_query(query, conn)

def main():
    # Create connection
    conn = create_connection()
    if conn is None:
        return
    
    # Setup database
    cursor = setup_database(conn)
    if cursor is None:
        return
    
    # Process data and update database
    process_data(conn)
    
    # Get and display sample results
    results = get_sample_results(conn)
    print("\nSample Results:")
    print(results)
    
    # Close connection
    conn.close()

if __name__ == "__main__":
    main()

C:\Users\twans\AppData\Local\Temp\ipykernel_42580\1818695058.py:139: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  starters = starters.groupby(['SEASON', 'START_POSITION']).apply(calculate_historic_avg)
C:\Users\twans\AppData\Local\Temp\ipykernel_42580\1818695058.py:144: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bench = bench.groupby('SEASON').apply(calculate_historic_avg)
